# Implementation of Time Series Data Mining for UI logs

We implement a time series motif discovery data mining approach for high noise userinteraction logs. These logs are:
- continuous recordings of users over extended periods of time
- contain data and context parameters, where data parameters change and context parameters remain consisten for every routine execution.
- the routines can be of varying length 
- there are multiple different potential routine candidates in the log

In [27]:
import sys
sys.path.append('../') # To import from parent dir
import pandas as pd
import numpy as np
import stumpy
import datetime
import time
import os
import math
import util.valmod_uihe as valmod_util
import util.util as util

In [28]:
def discover_joint_motifs_full(X, min_m, max_m, top_k=5):
    """
    Computes Joint Matrix Profile across all dimensions.
    X: Multivariate matrix (T x D)
    """
    data = X.T # Transpose for STUMPY (Dimensions x Time)
    found_motifs = []
    
    for m in range(min_m, max_m + 1):
        # Start of Joint Calculation
        joint_sq_dist = None
        for d in range(data.shape[0]):
            # STUMPY stump handles Z-normalization (Keogh's requirement)
            mp = stumpy.stump(data[d], m=m)
            # mp is dtype=object; first column is the matrix profile distances
            mp_dist = mp[:, 0].astype(np.float64)
            if joint_sq_dist is None:
                joint_sq_dist = mp_dist**2
            else:
                joint_sq_dist += mp_dist**2
        
        joint_mp = np.sqrt(joint_sq_dist)
        
        # Extract Top-K non-overlapping motifs
        temp_mp = joint_mp.copy()
        for k in range(top_k):
            best_idx = np.argmin(temp_mp)
            best_val = temp_mp[best_idx]
            
            if np.isinf(best_val): break
            
            found_motifs.append({
                'start_index': best_idx,
                'end_index': best_idx + m,
                'length': m,
                'distance': best_val,
                'rank': k + 1
            })
            
            # Apply exclusion zone to find distinct routines
            zone_start = max(0, best_idx - m)
            zone_end = min(len(temp_mp), best_idx + m)
            temp_mp[zone_start:zone_end] = np.inf
            
    return pd.DataFrame(found_motifs)

def discover_joint_motifs_optimized(X, min_m, max_m, top_k=5):
    # X is (T, D), STUMPY multivariate expects (D, T)
    data = X.T 
    found_motifs = []
    
    for m in range(min_m, max_m + 1):
        # mstump computes distances for all dimension combinations.
        # The last row (index -1) includes ALL dimensions equally.
        mps, indices = stumpy.mstump(data, m=m)
        joint_mp = mps[-1] 
        
        # Top-K extraction (this part is already quite efficient)
        temp_mp = joint_mp.copy()
        for k in range(top_k):
            best_idx = np.argmin(temp_mp)
            best_val = temp_mp[best_idx]
            if np.isinf(best_val): break
            
            found_motifs.append({
                'start_index': best_idx,
                'end_index': best_idx + m,
                'length': m,
                'distance': best_val,
                'rank': k + 1
            })
            
            # Exclusion zone
            zone_start = max(0, best_idx - m)
            zone_end = min(len(temp_mp), best_idx + m)
            temp_mp[zone_start:zone_end] = np.inf
            
    return pd.DataFrame(found_motifs)

In [30]:
# ============================================================
# CASE SELECTION  — switch between SmartRPA2025 and Leno data
# ============================================================
# Available cases:
#   "smartrpa2025"                   – synthetic SmartRPA2025 log (single routine type)
#   "leno_sr_rt_plus"                – Leno, sequential SR+RT, no noise
#   "leno_sr_rt_parallel"            – Leno, parallel SR||RT, no noise
#   "leno_sr_rt_plus_extended"       – Leno, sequential, 50 motifs w/ intra-motif noise
#   "leno_sr_rt_parallel_extended"   – Leno, parallel,   50 motifs w/ intra-motif noise

CASE = "smartrpa2025"   # <-- Change this value only

LENO_CASES = {
    "leno_sr_rt_plus":              ("202511_SR_RT_plus.csv",
                                     "202511_SR_RT_plus_ground_truth.csv"),
    "leno_sr_rt_parallel":          ("202511_SR_RT_parallel.csv",
                                     "202511_SR_RT_parallel_ground_truth.csv"),
    "leno_sr_rt_plus_extended":     ("202511_SR_RT_plus_extended.csv",
                                     "202511_SR_RT_plus_extended_ground_truth.csv"),
    "leno_sr_rt_parallel_extended": ("202511_SR_RT_parallel_extended.csv",
                                     "202511_SR_RT_parallel_extended_ground_truth.csv"),
}

# Per-case algorithm window. Leno RT motifs run up to ~63 events; SR ~34 events.
# SmartRPA2025 synthetic motifs are length 20 by construction.
CASE_PARAMS = {
    "smartrpa2025":                 {"min_m": 5,  "max_m": 60, "top_k": 100},
    "leno_sr_rt_plus":              {"min_m": 5,  "max_m": 65, "top_k": 60},
    "leno_sr_rt_parallel":          {"min_m": 5,  "max_m": 65, "top_k": 60},
    "leno_sr_rt_plus_extended":     {"min_m": 5,  "max_m": 70, "top_k": 60},
    "leno_sr_rt_parallel_extended": {"min_m": 5,  "max_m": 70, "top_k": 60},
}

if CASE not in CASE_PARAMS:
    raise ValueError(f"Unknown CASE '{CASE}'. Valid values: {list(CASE_PARAMS.keys())}")

# SmartRPA log name used only if CASE == 'smartrpa2025'
log_name_smartRPA = 'log_motifs2_occurances15_length20_percentage10_shuffle0.2.csv'

isSmartRPA2025  = (CASE == "smartrpa2025")
isActionLogger  = CASE.startswith("leno_")
leno_file_name, leno_gt_file_name = LENO_CASES.get(CASE, ("", ""))

encoding_method = 1
printing        = True
min_m  = CASE_PARAMS[CASE]["min_m"]
max_m  = CASE_PARAMS[CASE]["max_m"]
top_k  = CASE_PARAMS[CASE]["top_k"]

print(f"Running case : {CASE}")
print(f"min_m={min_m}, max_m={max_m}, top_k={top_k}")

# Step 2: Read data for processing, assign to variables
data_for_processing = util.read_data_for_processing(
    isSmartRPA2024=False,
    isSmartRPA2025=isSmartRPA2025,
    isActionLogger=isActionLogger,
    leno_file_name=leno_file_name,
    leno_gt_file_name=leno_gt_file_name,
    log_name_smartRPA=log_name_smartRPA,
)

hierarchy_list = data_for_processing["hierarchy_list"]
hierarchy_columns = data_for_processing["hierarchy_columns"]
hierarchy_columns_app_switch = data_for_processing["hierarchy_columns_app_switch"]
file = data_for_processing["file"]
log = data_for_processing["log"]
ground_truth = data_for_processing["ground_truth"]

#Step 3: Generate Word2vec embeddings for the log data
#Step 3.1: Calculate the vector size for token based encoding methods (Word2Vec)
hierarchy_columns = [
    col for col in hierarchy_columns
    if log[col].nunique() != 0
]
tokens = 0
for col in hierarchy_columns:
    tokens += log[col].nunique()
token_based_vector_size = round(math.sqrt(tokens))

#Step 3.2: Encode the filtered log
if encoding_method == 0:
    if printing:
        print("Using Word2Vec Sentence based encoding for UI Log")
    log_encoded = valmod_util.encode_word2vec_row_as_sentence(uiLog=log, 
                                                    orderedColumnsList=hierarchy_columns, 
                                                    window=len(hierarchy_columns),
                                                    vector_size=token_based_vector_size,
                                                    completeCorpusLog=log)
    column_identifier = 'w2v_'
elif encoding_method == 1:
    if printing:
        print("Using Word2Vec Detailed (Attribute as Word, Row as Sentence, Log as Corpus) based encoding for UI Log with vector size:", token_based_vector_size)
    log_encoded = valmod_util.encode_word2vec(uiLog=log, 
                                                        orderedColumnsList=hierarchy_columns, 
                                                        vector_size=token_based_vector_size,
                                                        completeCorpusLog=log)
    column_identifier = 'w2v_'

# Step 4: Apply PCA to the encoded log data
# stumpy requires float64
X = log_encoded.filter(like="w2v_").to_numpy().astype(np.float64)
# 4. Run Joint Discovery
start_time = time.time()
motifs_df = discover_joint_motifs_full(X, min_m, max_m, top_k=top_k)
duration = time.time() - start_time
print(f"Discovery duration: {duration:.1f}s")

Running case : smartrpa2025
min_m=5, max_m=60, top_k=100
Processing file: log_motifs2_occurances15_length20_percentage10_shuffle0.2.csv with 6000 events.
Using Word2Vec Detailed (Attribute as Word, Row as Sentence, Log as Corpus) based encoding for UI Log with vector size: 22
Discovery duration: 46.8s


In [ ]:
def drop_contained_motifs(df: pd.DataFrame, overlap_frac: float = 0.5) -> pd.DataFrame:
    """
    Pairwise dominance filter on overlap.

    For every pair (A, B) with span overlap O:
      - drop A when O / length_A >= overlap_frac AND distance_B <  distance_A
      - drop A when O / length_A >= overlap_frac AND distance_B == distance_A
        AND length_A < length_B  (equal-distance tiebreak: prefer the longer span)
      - symmetric rule for dropping B

    overlap_frac = 1.0 with tiebreak on full containment is the classic rule.
    overlap_frac = 0.5 also catches near-duplicate sub-fragments that aren't
    strictly contained (e.g. a length-11 d=0 motif sharing 10 events with a
    length-20 d=0 motif, but shifted by one position).
    """
    if df.empty:
        return df.copy()

    a = df.reset_index().rename(
        columns=lambda c: f"{c}_a" if c != "index" else "idx_a"
    )
    b = df.reset_index().rename(
        columns=lambda c: f"{c}_b" if c != "index" else "idx_b"
    )
    paired = a.merge(b, how="cross")
    paired = paired[paired["idx_a"] != paired["idx_b"]].copy()

    overlap = (
        np.minimum(paired["end_index_a"],   paired["end_index_b"]) -
        np.maximum(paired["start_index_a"], paired["start_index_b"])
    ).clip(lower=0)

    frac_a_in_b = overlap / paired["length_a"]
    frac_b_in_a = overlap / paired["length_b"]

    drop_a_strict = (frac_a_in_b >= overlap_frac) & (paired["distance_b"] < paired["distance_a"])
    drop_b_strict = (frac_b_in_a >= overlap_frac) & (paired["distance_a"] < paired["distance_b"])

    # Equal-distance tiebreak: now fires on overlap_frac, not only on full containment.
    drop_a_tie = (
        (frac_a_in_b >= overlap_frac) &
        (paired["distance_a"] == paired["distance_b"]) &
        (paired["length_a"]   <  paired["length_b"])
    )

    dominated_idx = pd.unique(
        pd.concat([
            paired.loc[drop_a_strict | drop_a_tie, "idx_a"],
            paired.loc[drop_b_strict, "idx_b"],
        ])
    )
    return df.drop(index=dominated_idx).reset_index(drop=True)

motifs_df_filtered = drop_contained_motifs(motifs_df, overlap_frac=0.5)
print(f"Before: {len(motifs_df)} rows | After: {len(motifs_df_filtered)} rows")
motifs_df_filtered

Before: 5341 rows | After: 257 rows


,start_index,end_index,length,distance,rank
0,546,551,5,0.000000,23
1,586,591,5,0.000000,24
2,916,921,5,0.000000,34
3,1121,1126,5,0.000000,40
4,1274,1279,5,0.000000,41
...,...,...,...,...,...
252,1614,1653,39,26.648276,73
253,1184,1223,39,26.907970,98
254,5507,5548,41,28.105760,99
255,4891,4936,45,30.011393,82


In [ ]:
def evaluate_motifs_vs_ground_truth(
    found: pd.DataFrame,
    ground_truth: pd.DataFrame,
    iou_threshold: float = 0.8,
):
    """
    Match each found motif to the ground-truth occurrence with the highest IoU.
    A match counts as a true positive only if IoU >= iou_threshold.

    Conventions assumed:
      - found:        columns start_index, end_index (end_index = start + length)
      - ground_truth: columns start_index, end_index (end_index inclusive per the
                      sample; treated half-open here for IoU math, which matches
                      how `found` is built in discover_joint_motifs_full).

    Returns: (annotated_found, summary_dict)
    """
    f = found.reset_index(drop=True).copy()
    g = ground_truth.reset_index(drop=True).copy()

    # Broadcast pairwise spans: shape (n_found, n_gt)
    f_start = f["start_index"].to_numpy()[:, None]
    f_end   = f["end_index"].to_numpy()[:, None]
    g_start = g["start_index"].to_numpy()[None, :]
    g_end   = g["end_index"].to_numpy()[None, :]

    inter = np.clip(np.minimum(f_end, g_end) - np.maximum(f_start, g_start), 0, None)
    union = (f_end - f_start) + (g_end - g_start) - inter
    iou   = np.where(union > 0, inter / union, 0.0)

    best_gt   = iou.argmax(axis=1)
    best_iou  = iou.max(axis=1)

    f["matched_gt_idx"] = best_gt
    f["iou"]            = best_iou
    f["is_tp"]          = best_iou >= iou_threshold

    # Recall: a GT occurrence counts as found if ANY predicted span hits it at >= threshold
    gt_hit = (iou >= iou_threshold).any(axis=0)

    tp = int(f["is_tp"].sum())
    fp = int(len(f) - tp)
    fn = int((~gt_hit).sum())
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall    = tp / (tp + fn) if (tp + fn) else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

    summary = {
        "iou_threshold": iou_threshold,
        "n_found": len(f),
        "n_ground_truth": len(g),
        "true_positives": tp,
        "false_positives": fp,
        "false_negatives": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "gt_coverage": gt_hit.mean() if len(g) else 0.0,
    }
    return f, summary


motifs_df_eval, summary = evaluate_motifs_vs_ground_truth(
    motifs_df_filtered, ground_truth, iou_threshold=0.8
)

print("=== Summary @ IoU >= 0.8 ===")
for k, v in summary.items():
    print(f"  {k}: {v}")
print("\n=== Per-found-motif (top matches) ===")
motifs_df_eval.sort_values("iou", ascending=False).head(50)

=== Summary @ IoU >= 0.8 ===
  iou_threshold: 0.8
  n_found: 257
  n_ground_truth: 30
  true_positives: 30
  false_positives: 227
  false_negatives: 0
  precision: 0.11673151750972763
  recall: 1.0
  f1: 0.20905923344947736
  gt_coverage: 1.0

=== Per-found-motif (top matches) ===


,start_index,end_index,length,distance,rank,matched_gt_idx,iou,is_tp
193,5647,5667,20,0.000000,29,20,0.95,True
178,2205,2225,20,0.000000,14,13,0.95,True
166,99,119,20,0.000000,2,15,0.95,True
167,124,144,20,0.000000,3,26,0.95,True
168,431,451,20,0.000000,4,22,0.95,True
169,490,510,20,0.000000,5,18,0.95,True
170,601,621,20,0.000000,6,25,0.95,True
171,650,670,20,0.000000,7,8,0.95,True
172,1047,1067,20,0.000000,8,12,0.95,True
173,1346,1366,20,0.000000,9,2,0.95,True


In [ ]:
motifs_df_eval.to_csv("motifs_eval.csv", index=False)